# AMR Resistance Mechanisms Analysis

This notebook analyzes antimicrobial resistance (AMR) genes grouped by their **resistance mechanism**.

## Mechanism Categories:
- **Drug Inactivation** - Enzymes that destroy/modify antibiotics (e.g., beta-lactamases)
- **Efflux Pump** - Pumps that expel antibiotics from the cell
- **Target Modification** - Changes to the antibiotic's target site
- **Target Protection** - Proteins that protect the target
- **Target Bypass** - Alternative enzymes that bypass the blocked target
- **Target Mutation** - Mutations in target genes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries loaded successfully!")

## 1. Define Mechanism Mapping

AMR genes are classified by their resistance mechanism based on gene name prefixes.

In [ ]:
# AMR Gene to Mechanism Mapping
MECHANISM_MAPPING = {
    # Beta-lactamases (Drug Inactivation)
    'bla': 'Beta-lactamase (Drug Inactivation)',
    
    # Aminoglycoside-modifying enzymes
    'aac': 'Aminoglycoside Acetyltransferase (Drug Inactivation)',
    'aad': 'Aminoglycoside Adenylyltransferase (Drug Inactivation)',
    'ant': 'Aminoglycoside Nucleotidyltransferase (Drug Inactivation)',
    'aph': 'Aminoglycoside Phosphotransferase (Drug Inactivation)',
    'armA': 'Aminoglycoside Methyltransferase (Target Modification)',
    'rmtA': 'Aminoglycoside Methyltransferase (Target Modification)',
    'rmtB': 'Aminoglycoside Methyltransferase (Target Modification)',
    
    # Efflux pumps
    'tet': 'Tetracycline Efflux/Ribosomal Protection',
    'oqx': 'RND Efflux Pump (Quinolone/Multidrug)',
    'qep': 'Quinolone Efflux Pump',
    'mef': 'Macrolide Efflux Pump',
    'msr': 'Macrolide Efflux Pump',
    'floR': 'Chloramphenicol Efflux Pump',
    'cml': 'Chloramphenicol Efflux Pump',
    'cat': 'Chloramphenicol Acetyltransferase (Drug Inactivation)',
    'qac': 'Quaternary Ammonium Compound Efflux',
    
    # Target modification/protection
    'mcr': 'Colistin Resistance (Target Modification)',
    'erm': 'Ribosome Methylase (Target Modification)',
    'cfr': 'Ribosome Methylase (Target Modification)',
    'mph': 'Macrolide Phosphotransferase (Drug Inactivation)',
    
    # Quinolone resistance
    'qnr': 'Quinolone Resistance (Target Protection)',
    
    # Folate pathway inhibitors
    'dfr': 'Dihydrofolate Reductase (Target Bypass)',
    'sul': 'Sulfonamide Resistance (Target Bypass)',
    
    # Glycopeptide resistance
    'van': 'Glycopeptide Resistance (Target Modification)',
    
    # Fosfomycin resistance
    'fos': 'Fosfomycin Resistance (Drug Inactivation)',
}

# Broader categories
MECHANISM_CATEGORIES = {
    'Drug Inactivation': [
        'Beta-lactamase (Drug Inactivation)',
        'Aminoglycoside Acetyltransferase (Drug Inactivation)',
        'Aminoglycoside Adenylyltransferase (Drug Inactivation)',
        'Aminoglycoside Nucleotidyltransferase (Drug Inactivation)',
        'Aminoglycoside Phosphotransferase (Drug Inactivation)',
        'Chloramphenicol Acetyltransferase (Drug Inactivation)',
        'Macrolide Phosphotransferase (Drug Inactivation)',
        'Fosfomycin Resistance (Drug Inactivation)',
    ],
    'Efflux Pump': [
        'Tetracycline Efflux/Ribosomal Protection',
        'RND Efflux Pump (Quinolone/Multidrug)',
        'Quinolone Efflux Pump',
        'Macrolide Efflux Pump',
        'Chloramphenicol Efflux Pump',
        'Quaternary Ammonium Compound Efflux',
    ],
    'Target Modification': [
        'Aminoglycoside Methyltransferase (Target Modification)',
        'Colistin Resistance (Target Modification)',
        'Ribosome Methylase (Target Modification)',
        'Glycopeptide Resistance (Target Modification)',
    ],
    'Target Protection': [
        'Quinolone Resistance (Target Protection)',
    ],
    'Target Bypass': [
        'Dihydrofolate Reductase (Target Bypass)',
        'Sulfonamide Resistance (Target Bypass)',
    ],
}

def get_mechanism(gene_name):
    """Determine resistance mechanism from gene name."""
    gene_lower = gene_name.lower()
    for prefix, mechanism in MECHANISM_MAPPING.items():
        if gene_lower.startswith(prefix.lower()):
            return mechanism
    if 'bla' in gene_lower:
        return 'Beta-lactamase (Drug Inactivation)'
    return 'Other/Unknown Mechanism'

def get_mechanism_category(mechanism):
    """Get broad category for a specific mechanism."""
    for category, mechanisms in MECHANISM_CATEGORIES.items():
        if mechanism in mechanisms:
            return category
    return 'Other'

print("Mechanism mapping defined!")

## 2. Load and Process Data

In [ ]:
# Load AMR data
df = pd.read_csv('amr_full_data.csv')
print(f"Loaded {len(df):,} AMR records")

# Add mechanism columns
df['mechanism'] = df['Gene'].apply(get_mechanism)
df['mechanism_category'] = df['mechanism'].apply(get_mechanism_category)

# Get unique gene detections (not expanded by resistance class)
unique_genes = df.drop_duplicates(subset=['Gene', 'ReadID', 'barcode'])
print(f"Unique gene detections: {len(unique_genes):,}")
print(f"Unique mechanisms: {df['mechanism'].nunique()}")
print(f"Unique categories: {df['mechanism_category'].nunique()}")

df.head()

## 3. Mechanism Category Distribution

In [ ]:
# Category distribution
category_counts = unique_genes['mechanism_category'].value_counts()

# Pie chart
fig = px.pie(
    values=category_counts.values,
    names=category_counts.index,
    title='AMR Resistance Mechanism Categories',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [ ]:
# Bar chart with counts
fig, ax = plt.subplots(figsize=(10, 6))

colors = sns.color_palette('husl', len(category_counts))
bars = ax.barh(category_counts.index, category_counts.values, color=colors)

# Add count labels
for bar, count in zip(bars, category_counts.values):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2, 
            f'{count:,}', va='center', fontsize=10)

ax.set_xlabel('Number of Gene Detections')
ax.set_title('AMR Resistance Mechanism Categories')
plt.tight_layout()
plt.show()

# Print percentages
print("\nCategory Distribution:")
for cat, count in category_counts.items():
    pct = count / len(unique_genes) * 100
    print(f"  {cat}: {count:,} ({pct:.1f}%)")

## 4. Top Specific Mechanisms

In [ ]:
# Top 10 specific mechanisms
mechanism_counts = unique_genes['mechanism'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 6))

colors = sns.color_palette('viridis', len(mechanism_counts))
bars = ax.barh(mechanism_counts.index[::-1], mechanism_counts.values[::-1], color=colors[::-1])

# Add count labels
for bar, count in zip(bars, mechanism_counts.values[::-1]):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2, 
            f'{count:,}', va='center', fontsize=9)

ax.set_xlabel('Number of Gene Detections')
ax.set_title('Top 10 AMR Resistance Mechanisms')
plt.tight_layout()
plt.show()

## 5. Mechanisms by Farm Type (Pig vs Poultry)

In [ ]:
# Category counts by farm type
farm_category = unique_genes.groupby(['farm_type', 'mechanism_category']).size().unstack(fill_value=0)

# Normalize to percentage
farm_category_pct = farm_category.div(farm_category.sum(axis=1), axis=0) * 100

# Stacked bar chart
fig, ax = plt.subplots(figsize=(10, 6))
farm_category_pct.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')

ax.set_ylabel('Percentage (%)')
ax.set_xlabel('Farm Type')
ax.set_title('AMR Mechanism Categories by Farm Type')
ax.legend(title='Mechanism Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Print raw counts
print("\nRaw counts by farm type:")
print(farm_category)

In [ ]:
# Interactive grouped bar chart
farm_melt = farm_category.reset_index().melt(id_vars='farm_type', 
                                              var_name='mechanism_category', 
                                              value_name='count')

fig = px.bar(
    farm_melt,
    x='mechanism_category',
    y='count',
    color='farm_type',
    barmode='group',
    title='AMR Mechanism Categories: Pig vs Poultry',
    labels={'count': 'Gene Detections', 'mechanism_category': 'Mechanism Category'},
    color_discrete_map={'Pig': '#1f77b4', 'Poultry': '#ff7f0e'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 6. Mechanism Heatmap by Sample

In [ ]:
# Create mechanism × sample matrix
mechanism_matrix = unique_genes.pivot_table(
    index='mechanism_category',
    columns='barcode',
    values='ReadID',
    aggfunc='count',
    fill_value=0
)

# Log transform for better visualization
mechanism_matrix_log = np.log10(mechanism_matrix + 1)

# Heatmap
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(mechanism_matrix_log, cmap='YlOrRd', annot=False, 
            cbar_kws={'label': 'log10(count + 1)'}, ax=ax)
ax.set_title('AMR Mechanism Categories by Sample (log10 scale)')
ax.set_xlabel('Sample (Barcode)')
ax.set_ylabel('Mechanism Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Interactive heatmap with Plotly
fig = px.imshow(
    mechanism_matrix_log,
    labels=dict(x='Barcode', y='Mechanism Category', color='log10(count+1)'),
    title='AMR Mechanism Categories by Sample',
    color_continuous_scale='YlOrRd',
    aspect='auto'
)
fig.show()

## 7. Statistical Comparison: Pig vs Poultry

In [ ]:
# Calculate mechanism category counts per sample
sample_mechanisms = unique_genes.groupby(['barcode', 'farm_type', 'mechanism_category']).size().unstack(fill_value=0)
sample_mechanisms = sample_mechanisms.reset_index()

# Get category columns
category_cols = [col for col in sample_mechanisms.columns if col not in ['barcode', 'farm_type']]

# Mann-Whitney U test for each category
print("Mann-Whitney U Test: Pig vs Poultry\n")
print(f"{'Category':<25} {'Pig Mean':>10} {'Poultry Mean':>12} {'U-stat':>10} {'p-value':>10}")
print("-" * 70)

results = []
for cat in category_cols:
    pig_vals = sample_mechanisms[sample_mechanisms['farm_type'] == 'Pig'][cat]
    poultry_vals = sample_mechanisms[sample_mechanisms['farm_type'] == 'Poultry'][cat]
    
    if len(pig_vals) > 0 and len(poultry_vals) > 0:
        stat, pval = stats.mannwhitneyu(pig_vals, poultry_vals, alternative='two-sided')
        sig = '*' if pval < 0.05 else ''
        print(f"{cat:<25} {pig_vals.mean():>10.1f} {poultry_vals.mean():>12.1f} {stat:>10.1f} {pval:>9.4f} {sig}")
        results.append({'category': cat, 'pig_mean': pig_vals.mean(), 
                       'poultry_mean': poultry_vals.mean(), 'p_value': pval})

print("\n* p < 0.05")

In [ ]:
# Box plots for each category
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, cat in enumerate(category_cols[:6]):
    ax = axes[i]
    data_to_plot = [sample_mechanisms[sample_mechanisms['farm_type'] == 'Pig'][cat],
                    sample_mechanisms[sample_mechanisms['farm_type'] == 'Poultry'][cat]]
    
    bp = ax.boxplot(data_to_plot, labels=['Pig', 'Poultry'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#1f77b4')
    bp['boxes'][1].set_facecolor('#ff7f0e')
    
    ax.set_title(cat, fontsize=9)
    ax.set_ylabel('Count')

plt.suptitle('AMR Mechanism Categories by Farm Type', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Gene-to-Mechanism Mapping Table

In [ ]:
# Create gene-mechanism mapping
gene_mapping = df[['Gene', 'mechanism', 'mechanism_category']].drop_duplicates()
gene_mapping = gene_mapping.sort_values(['mechanism_category', 'mechanism', 'Gene'])

print(f"Total unique genes: {len(gene_mapping)}")
print("\nGenes per mechanism category:")
print(gene_mapping['mechanism_category'].value_counts())

# Show sample
gene_mapping.head(20)

## 9. Mechanism Diversity per Sample

In [ ]:
# Calculate diversity metrics per sample
diversity = unique_genes.groupby(['barcode', 'farm_type']).agg({
    'Gene': 'nunique',
    'mechanism': 'nunique',
    'mechanism_category': 'nunique'
}).reset_index()

diversity.columns = ['barcode', 'farm_type', 'unique_genes', 'unique_mechanisms', 'unique_categories']

# Box plot
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for i, col in enumerate(['unique_genes', 'unique_mechanisms', 'unique_categories']):
    ax = axes[i]
    data_to_plot = [diversity[diversity['farm_type'] == 'Pig'][col],
                    diversity[diversity['farm_type'] == 'Poultry'][col]]
    
    bp = ax.boxplot(data_to_plot, labels=['Pig', 'Poultry'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#1f77b4')
    bp['boxes'][1].set_facecolor('#ff7f0e')
    
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Count')

plt.suptitle('AMR Diversity by Farm Type', fontsize=12)
plt.tight_layout()
plt.show()

# Summary statistics
print("\nDiversity Summary:")
print(diversity.groupby('farm_type')[['unique_genes', 'unique_mechanisms', 'unique_categories']].describe())

## 10. Summary Statistics

In [ ]:
print("=" * 50)
print("AMR MECHANISM ANALYSIS SUMMARY")
print("=" * 50)

print(f"\nTotal AMR records: {len(df):,}")
print(f"Unique gene detections: {len(unique_genes):,}")
print(f"Unique resistance genes: {df['Gene'].nunique()}")
print(f"Unique mechanisms: {df['mechanism'].nunique()}")
print(f"Unique mechanism categories: {df['mechanism_category'].nunique()}")

print("\nSamples:")
print(f"  Total: {df['barcode'].nunique()}")
print(f"  Pig farms: {len(df[df['farm_type'] == 'Pig']['barcode'].unique())}")
print(f"  Poultry farms: {len(df[df['farm_type'] == 'Poultry']['barcode'].unique())}")

print("\nMechanism Category Distribution:")
for cat, count in category_counts.items():
    pct = count / len(unique_genes) * 100
    print(f"  {cat}: {count:,} ({pct:.1f}%)")